# ReXErr error detection — GPU experiments

**Kaggle (recommended):** Settings → Accelerator **GPU T4 x2**, Internet **on** (needs a verified phone number), then
**Save Version → Save & Run All (Commit)**; the run continues in the background and its files appear in the Output tab.
Each Kaggle session is limited to 12 hours, so run the groups in separate versions:

| Version | `RUN_GROUPS` | Accelerator | Approx. time |
|---|---|---|---|
| 1 | `["main", "encoders"]` — BioBERT × 3 seeds; BERT-base-cased, Bio+ClinicalBERT, BiomedBERT, BlueBERT, CXR-BERT, RadBERT | GPU T4 x2 | ~6.5 h |
| 2 | `["extras"]` — localization, no-control ablation, weighted-BCE and focal-loss BioBERT, Qwen2.5-7B, Llama-3.1-8B and Qwen3-8B zero-shot | GPU T4 x2 | ~4 h |
| 3 | `["tfidf"]` — full-data TF-IDF baseline (needs > 24 GB RAM, no GPU) | None (CPU) | ~1.5 h |

Each version writes `rexerr_results.zip` (prediction files only, no report text) to its Output tab. Runs already finished
in an earlier version are skipped if that version's output is added as an input (Add Input → Your Work).

**Colab:** choose a GPU runtime and Run all; results go to Google Drive and re-running resumes where it stopped.
Times are estimates.


In [ ]:
# Choose which groups to run in this session: "main", "encoders", "extras", "tfidf".
RUN_GROUPS = ["main", "encoders"]
EPOCHS = 2
BATCH_SIZE = 32            # effective batch, identical on every GPU setup so runs are comparable
UNASSIGN_WHEN_DONE = True  # Colab only: release the runtime after the last cell
# Optional "owner/slug" of a Kaggle dataset holding the ReXErr-v1 zip; downloaded with the KAGGLE_API_TOKEN
# secret (Colab: key icon) or environment variable instead of the slower PhysioNet server.
KAGGLE_DATASET = ""


In [ ]:
import glob, os, shutil, subprocess, sys
IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = os.path.exists("/kaggle")
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = "/content/drive/MyDrive/rexerr_experiments"
    CACHE = WORK                 # keep the 87 MB archive on Drive between sessions
elif IN_KAGGLE:
    WORK = "/kaggle/working/rexerr_experiments"  # saved as notebook output
    CACHE = "/tmp"
else:
    WORK = os.path.abspath("rexerr_experiments")
    CACHE = WORK
DATA = "/tmp/rexerr_data"        # local disk, not saved: fast reads and a small output
CODE, RUNS = f"{WORK}/code", f"{WORK}/runs"
for d in (CODE + "/rexerr", CODE + "/scripts", DATA, RUNS):
    os.makedirs(d, exist_ok=True)
if IN_KAGGLE:  # reuse runs finished in earlier versions whose output was added as an input
    for f in glob.glob("/kaggle/input/**/runs/*/preds_test.npz", recursive=True):
        src = os.path.dirname(f)
        dst = f"{RUNS}/{os.path.basename(src)}"
        if not os.path.exists(f"{dst}/preds_test.npz"):
            shutil.copytree(src, dst, dirs_exist_ok=True)
            print("reusing finished run:", os.path.basename(src))
import torch
N_GPU = max(1, torch.cuda.device_count())  # T4 x2: per-device batch is halved to keep the effective batch at 32
GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"
GPU_MEM_TOTAL = sum(torch.cuda.get_device_properties(i).total_memory for i in range(torch.cuda.device_count()))
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("WORK =", WORK)
print(f"GPU: {GPU_NAME} x{torch.cuda.device_count()} ({GPU_MEM_TOTAL / 1e9:.0f} GB in total)")
if not torch.cuda.is_available() and set(RUN_GROUPS) - {"tfidf"}:
    print("NO GPU FOUND - Kaggle: Settings > Accelerator > GPU T4 x2; Colab: Runtime > Change runtime type")


In [ ]:
!pip -q install "transformers==5.20.1" "accelerate>=1.0" "scikit-learn>=1.4" "pandas>=2.0" "scipy" "bitsandbytes>=0.45"


### Source code (embedded)


In [ ]:
%%writefile {CODE}/rexerr/__init__.py
"""Error detection in radiology reports: ReXErr-v1 experiments."""


In [ ]:
%%writefile {CODE}/rexerr/data.py
"""Loading ReXErr-v1 and building the report-level and sentence-level tasks.

Design decisions (see README):
  * Official ReXErr/MIMIC-CXR train/val/test splits are used (patient-disjoint).
  * Each study contributes its error-injected report (labels = error types) and,
    when ``include_clean`` is set, its original report as an error-free
    negative (all-zero labels).
  * Primary labels come from the sentence-level annotations, which the dataset
    authors recommend over ``errors_sampled`` (sampled errors are not always
    present in the generated report).
  * Whitespace is collapsed because GPT-4o rewrites spacing, which would let a
    model separate original from error reports by formatting alone.
"""
import os
import re

import numpy as np
import pandas as pd

from .labels import NATIVE, normalize_type, parse_sampled

SPLITS = ("train", "val", "test")


def normalize_text(text):
    if not isinstance(text, str):
        return ""
    return re.sub(r"\s+", " ", text).strip()


def report_path(data_dir, split):
    return os.path.join(data_dir, "ReXErr-report-level", f"ReXErr-report-level_{split}.csv")


def sentence_path(data_dir, split):
    return os.path.join(data_dir, "ReXErr-sentence-level", f"ReXErr-sentence-level_{split}.csv")


def load_reports(data_dir, split):
    return pd.read_csv(report_path(data_dir, split))


def load_sentences(data_dir, split):
    return pd.read_csv(sentence_path(data_dir, split))


def sentence_label_sets(sent_df):
    """study_id -> set of native error types annotated at sentence level."""
    err = sent_df[sent_df["error_present"] == 1]
    out = {}
    for sid, t in zip(err["study_id"], err["error_type"]):
        key = normalize_type(t)
        if key:
            out.setdefault(sid, set()).add(key)
    return out


def to_multihot(label_sets):
    y = np.zeros((len(label_sets), len(NATIVE)), dtype=np.float32)
    idx = {k: i for i, k in enumerate(NATIVE)}
    for r, s in enumerate(label_sets):
        for k in s:
            y[r, idx[k]] = 1.0
    return y


def build_report_task(data_dir, split, label_source="sentence", include_clean=True,
                      max_studies=None, seed=0):
    """Return (frame, Y) for the report-level multi-label task.

    frame columns: uid, study_id, subject_id, variant ('error'|'original'), text
    Y: float32 array (n, 12) over ``labels.NATIVE``.
    """
    rep = load_reports(data_dir, split)
    if max_studies is not None and max_studies < len(rep):
        rng = np.random.default_rng(seed)
        subjects = rep["subject_id"].unique()
        rng.shuffle(subjects)
        keep, n = set(), 0
        counts = rep.groupby("subject_id").size()
        for s in subjects:
            if n >= max_studies:
                break
            keep.add(s)
            n += counts[s]
        rep = rep[rep["subject_id"].isin(keep)]

    if label_source == "sentence":
        sets = sentence_label_sets(load_sentences(data_dir, split))
        err_sets = [sets.get(s, set()) for s in rep["study_id"]]
    elif label_source == "sampled":
        err_sets = [set(parse_sampled(v)) for v in rep["errors_sampled"]]
    else:
        raise ValueError(label_source)

    rows, label_sets = [], []
    for (_, r), s in zip(rep.iterrows(), err_sets):
        err_text = normalize_text(r["error_report"])
        orig_text = normalize_text(r["original_report"])
        if err_text and err_text != orig_text:
            rows.append((f"{r.study_id}_e", r.study_id, r.subject_id, "error", err_text))
            label_sets.append(s)
        if include_clean and orig_text:
            rows.append((f"{r.study_id}_o", r.study_id, r.subject_id, "original", orig_text))
            label_sets.append(set())
    frame = pd.DataFrame(rows, columns=["uid", "study_id", "subject_id", "variant", "text"])
    return frame, to_multihot(label_sets)


def build_sentence_task(data_dir, split, include_clean=True, max_studies=None, seed=0):
    """Sentence-level localization task.

    Each example is (sentence, full report as context). Error sentences carry
    their error type; unchanged and 'neutral' (error_present == 2) sentences,
    and all sentences of the original report, are negatives.
    """
    sent = load_sentences(data_dir, split)
    rep = load_reports(data_dir, split).set_index("study_id")
    studies = sent["study_id"].unique()
    if max_studies is not None and max_studies < len(studies):
        rng = np.random.default_rng(seed)
        subj = sent.drop_duplicates("study_id").set_index("study_id")["subject_id"]
        subjects = subj.unique()
        rng.shuffle(subjects)
        keep, n = set(), 0
        per_subject = subj.groupby(subj).size()
        for s in subjects:
            if n >= max_studies:
                break
            keep.add(s)
            n += per_subject[s]
        sent = sent[sent["subject_id"].isin(keep)]

    rows, label_sets = [], []
    for sid, g in sent.groupby("study_id", sort=False):
        if sid not in rep.index:
            continue
        r = rep.loc[sid]
        err_ctx = normalize_text(r["error_report"])
        orig_ctx = normalize_text(r["original_report"])
        subject = int(g["subject_id"].iloc[0])
        for i, (es, ep, et) in enumerate(zip(g["error_sentence"], g["error_present"], g["error_type"])):
            es = normalize_text(es)
            if not es:
                continue
            key = normalize_type(et) if ep == 1 else None
            rows.append((f"{sid}_e{i}", sid, subject, "error", i, es, err_ctx, int(ep) if ep == ep else 0))
            label_sets.append({key} if key else set())
        if include_clean:
            for i, os_ in enumerate(g["original_sentence"]):
                os_ = normalize_text(os_)
                if not os_:
                    continue
                rows.append((f"{sid}_o{i}", sid, subject, "original", i, os_, orig_ctx, 0))
                label_sets.append(set())
    frame = pd.DataFrame(rows, columns=["uid", "study_id", "subject_id", "variant", "position",
                                        "text", "context", "error_present"])
    return frame, to_multihot(label_sets)


In [ ]:
%%writefile {CODE}/rexerr/io.py
"""Shared prediction-file format so that every model is evaluated by the same code."""
import json
import os

import numpy as np

from .labels import NATIVE


def save_predictions(out_dir, split, frame, y, p, meta=None):
    os.makedirs(out_dir, exist_ok=True)
    extra = {c: frame[c].to_numpy(dtype=np.int64) for c in ("position", "error_present") if c in frame}
    np.savez_compressed(
        os.path.join(out_dir, f"preds_{split}.npz"),
        uid=np.array(frame["uid"].astype(str).tolist(), dtype=str),
        study_id=frame["study_id"].to_numpy(dtype=np.int64), subject_id=frame["subject_id"].to_numpy(dtype=np.int64),
        variant=np.array(frame["variant"].astype(str).tolist(), dtype=str),
        y=np.asarray(y, dtype=np.int8), p=np.asarray(p, dtype=np.float32), classes=np.array(NATIVE), **extra,
    )
    if meta is not None:
        with open(os.path.join(out_dir, "meta.json"), "w") as f:
            json.dump(meta, f, indent=2, default=str)


def load_predictions(run_dir, split):
    d = np.load(os.path.join(run_dir, f"preds_{split}.npz"), allow_pickle=False)
    return {k: d[k] for k in d.files}


def load_meta(run_dir):
    path = os.path.join(run_dir, "meta.json")
    if not os.path.exists(path):
        return {}
    with open(path) as f:
        return json.load(f)


In [ ]:
%%writefile {CODE}/rexerr/labels.py
"""Error taxonomy for ReXErr-v1.

The 12 native ReXErr error types are the primary label space. The 6-group
grouping merges related types into six clinically motivated groups and is
reported as a secondary view.
"""
import ast
import re

# Native ReXErr-v1 error types (README order).
NATIVE = [
    "add_medical_device",
    "false_prediction",
    "false_negation",
    "add_repetitions",
    "add_contradictions",
    "change_name_of_device",
    "change_position_of_device",
    "change_severity",
    "change_location",
    "change_measurement",
    "change_to_homophone",
    "add_typo",
]

NATIVE_DISPLAY = {
    "add_medical_device": "Add medical device",
    "false_prediction": "False prediction",
    "false_negation": "False negation",
    "add_repetitions": "Add repetitions",
    "add_contradictions": "Add contradictions",
    "change_name_of_device": "Change name of device",
    "change_position_of_device": "Change position of device",
    "change_severity": "Change severity",
    "change_location": "Change location",
    "change_measurement": "Change measurement",
    "change_to_homophone": "Change to homophone",
    "add_typo": "Add typo",
}

# ReXErr's own three-way grouping (README).
REXERR_GROUP = {
    "add_medical_device": "content_addition",
    "false_prediction": "content_addition",
    "false_negation": "content_addition",
    "add_repetitions": "linguistic_quality",
    "add_contradictions": "linguistic_quality",
    "change_to_homophone": "linguistic_quality",
    "add_typo": "linguistic_quality",
    "change_name_of_device": "context_dependent",
    "change_position_of_device": "context_dependent",
    "change_severity": "context_dependent",
    "change_location": "context_dependent",
    "change_measurement": "context_dependent",
}

# Six-group mapping. The linguistic group holds typographical/transcription
# errors (typos, homophones, repetitions); contradictions in ReXErr are not
# restricted to findings versus impression.
GROUP6 = [
    "false_findings",
    "incorrect_location",
    "measurement_error",
    "internal_contradiction",
    "linguistic_error",
    "severity_error",
]

GROUP6_DISPLAY = {
    "false_findings": "False findings / devices",
    "incorrect_location": "Incorrect location / position",
    "measurement_error": "Measurement error",
    "internal_contradiction": "Internal contradiction",
    "linguistic_error": "Linguistic / transcription error",
    "severity_error": "Severity misclassification",
}

NATIVE_TO_GROUP6 = {
    "false_prediction": "false_findings",
    "false_negation": "false_findings",
    "add_medical_device": "false_findings",
    "change_name_of_device": "false_findings",
    "change_location": "incorrect_location",
    "change_position_of_device": "incorrect_location",
    "change_measurement": "measurement_error",
    "add_contradictions": "internal_contradiction",
    "add_repetitions": "linguistic_error",
    "change_to_homophone": "linguistic_error",
    "add_typo": "linguistic_error",
    "change_severity": "severity_error",
}


def normalize_type(name):
    """Map any spelling used in ReXErr files to a NATIVE key (or None)."""
    if name is None:
        return None
    s = str(name).strip().lower()
    if not s or s in {"nan", "not applicable", "none", "n/a"}:
        return None
    s = re.sub(r"[^a-z ]+", " ", s)
    s = re.sub(r"\s+", "_", s.strip())
    # Sentence-level files use singular forms ("Add repetition").
    s = {"add_repetition": "add_repetitions", "add_contradiction": "add_contradictions"}.get(s, s)
    return s if s in NATIVE else None


def parse_sampled(value):
    """Parse the `errors_sampled` column (a stringified Python list)."""
    if value is None or (isinstance(value, float) and value != value):
        return []
    try:
        items = ast.literal_eval(str(value))
    except (ValueError, SyntaxError):
        items = re.split(r"[,;]", str(value).strip("[]"))
    if isinstance(items, str):
        items = [items]
    return [t for t in (normalize_type(i) for i in items) if t]


def group6_matrix(y_native):
    """Collapse an (n, 12) native matrix into the (n, 6) group matrix (logical OR / max)."""
    import numpy as np

    y_native = np.asarray(y_native)
    out = np.zeros((y_native.shape[0], len(GROUP6)), dtype=y_native.dtype)
    for j, name in enumerate(NATIVE):
        g = GROUP6.index(NATIVE_TO_GROUP6[name])
        out[:, g] = np.maximum(out[:, g], y_native[:, j])
    return out


In [ ]:
%%writefile {CODE}/rexerr/metrics.py
"""Evaluation metrics, bootstrap confidence intervals and prevalence analysis.

All functions take y (n, K) binary labels and p (n, K) probabilities. Classes
without positives in the evaluated set are excluded from macro averages.
"""
import numpy as np
from scipy.stats import rankdata
from sklearn.metrics import average_precision_score


def _auroc(y, s):
    pos = y == 1
    n_pos, n_neg = pos.sum(), (~pos).sum()
    if n_pos == 0 or n_neg == 0:
        return np.nan
    r = rankdata(s)
    return (r[pos].sum() - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg)


def _auprc(y, s):
    if y.sum() == 0:
        return np.nan
    return average_precision_score(y, s)


def _prf(tp, fp, fn):
    p = tp / (tp + fp) if tp + fp else 0.0
    r = tp / (tp + fn) if tp + fn else 0.0
    f = 2 * p * r / (p + r) if p + r else 0.0
    return p, r, f


def as_thresholds(thr, k):
    return np.full(k, thr, dtype=float) if np.isscalar(thr) else np.asarray(thr, dtype=float)


def active_classes(y):
    return np.asarray(y).sum(axis=0) > 0


def per_class(y, p, names, thr=0.5):
    y, p = np.asarray(y).astype(int), np.asarray(p, dtype=float)
    t = as_thresholds(thr, y.shape[1])
    yhat = (p >= t).astype(int)
    rows = []
    for j, name in enumerate(names):
        tp = int(((yhat[:, j] == 1) & (y[:, j] == 1)).sum())
        fp = int(((yhat[:, j] == 1) & (y[:, j] == 0)).sum())
        fn = int(((yhat[:, j] == 0) & (y[:, j] == 1)).sum())
        tn = int(((yhat[:, j] == 0) & (y[:, j] == 0)).sum())
        pr, rc, f1 = _prf(tp, fp, fn)
        rows.append({
            "class": name, "support": int(y[:, j].sum()), "precision": pr, "recall": rc, "f1": f1,
            "specificity": tn / (tn + fp) if tn + fp else np.nan,
            "auroc": _auroc(y[:, j], p[:, j]), "auprc": _auprc(y[:, j], p[:, j]),
            "tp": tp, "fp": fp, "fn": fn, "tn": tn, "threshold": float(t[j]),
        })
    return rows


def summary(y, p, thr=0.5, mask=None):
    """Macro/micro/weighted F1, macro AUROC/AUPRC, Hamming and exact-match accuracy."""
    y, p = np.asarray(y).astype(int), np.asarray(p, dtype=float)
    if mask is None:
        mask = active_classes(y)
    y, p = y[:, mask], p[:, mask]
    t = as_thresholds(thr, len(mask))[mask]
    yhat = (p >= t).astype(int)
    tp = ((yhat == 1) & (y == 1)).sum(0)
    fp = ((yhat == 1) & (y == 0)).sum(0)
    fn = ((yhat == 0) & (y == 1)).sum(0)
    f1 = np.array([_prf(a, b, c)[2] for a, b, c in zip(tp, fp, fn)])
    support = y.sum(0)
    _, _, micro = _prf(tp.sum(), fp.sum(), fn.sum())
    au = np.array([_auroc(y[:, j], p[:, j]) for j in range(y.shape[1])])
    ap = np.array([_auprc(y[:, j], p[:, j]) for j in range(y.shape[1])])
    return {
        "f1_macro": float(f1.mean()),
        "f1_micro": float(micro),
        "f1_weighted": float((f1 * support).sum() / max(support.sum(), 1)),
        "auroc_macro": float(np.nanmean(au)),
        "auprc_macro": float(np.nanmean(ap)),
        "hamming_accuracy": float((yhat == y).mean()),
        "exact_match_accuracy": float((yhat == y).all(1).mean()),
        "n_classes": int(mask.sum()),
    }


def any_error_score(p):
    """Report-level probability that at least one error is present."""
    return np.asarray(p).max(axis=1)


def binary(y_any, s_any, thr=0.5):
    y_any, s_any = np.asarray(y_any).astype(int), np.asarray(s_any, dtype=float)
    yhat = (s_any >= thr).astype(int)
    tp = int(((yhat == 1) & (y_any == 1)).sum()); fp = int(((yhat == 1) & (y_any == 0)).sum())
    fn = int(((yhat == 0) & (y_any == 1)).sum()); tn = int(((yhat == 0) & (y_any == 0)).sum())
    pr, rc, f1 = _prf(tp, fp, fn)
    return {
        "precision": pr, "sensitivity": rc, "f1": f1,
        "specificity": tn / (tn + fp) if tn + fp else np.nan,
        "auroc": float(_auroc(y_any, s_any)), "auprc": float(_auprc(y_any, s_any)),
        "tp": tp, "fp": fp, "fn": fn, "tn": tn, "threshold": float(thr),
        "prevalence_in_set": float(y_any.mean()),
    }


def tune_thresholds(y_val, p_val, grid=None):
    """Per-class threshold maximising F1 on the validation set (0.5 if no positives)."""
    grid = np.linspace(0.05, 0.95, 91) if grid is None else grid
    y_val, p_val = np.asarray(y_val).astype(int), np.asarray(p_val, dtype=float)
    out = np.full(y_val.shape[1], 0.5)
    for j in range(y_val.shape[1]):
        if y_val[:, j].sum() == 0:
            continue
        best, best_t = -1, 0.5
        for t in grid:
            yhat = p_val[:, j] >= t
            tp = (yhat & (y_val[:, j] == 1)).sum(); fp = (yhat & (y_val[:, j] == 0)).sum()
            fn = (~yhat & (y_val[:, j] == 1)).sum()
            f = _prf(tp, fp, fn)[2]
            if f > best:
                best, best_t = f, t
        out[j] = best_t
    return out


def threshold_for_sensitivity(y_any, s_any, target):
    """Highest threshold whose sensitivity on (y_any, s_any) is >= target."""
    pos = np.sort(np.asarray(s_any)[np.asarray(y_any) == 1])[::-1]
    if len(pos) == 0:
        return 0.5
    k = int(np.ceil(target * len(pos))) - 1
    return float(pos[min(max(k, 0), len(pos) - 1)])


def prevalence_analysis(sens, spec, prevalences=(0.03, 0.05, 0.10, 0.20)):
    """Expected PPV/NPV and alert burden when the true error rate is `prevalence`."""
    rows = []
    for pi in prevalences:
        tp, fp = sens * pi, (1 - spec) * (1 - pi)
        fn, tn = (1 - sens) * pi, spec * (1 - pi)
        rows.append({
            "prevalence": pi,
            "ppv": tp / (tp + fp) if tp + fp else np.nan,
            "npv": tn / (tn + fn) if tn + fn else np.nan,
            "alerts_per_1000": 1000 * (tp + fp),
            "true_alerts_per_1000": 1000 * tp,
            "missed_per_1000": 1000 * fn,
        })
    return rows


def _group_index(groups):
    groups = np.asarray(groups)
    uniq, inv = np.unique(groups, return_inverse=True)
    order = np.argsort(inv, kind="stable")
    bounds = np.searchsorted(inv[order], np.arange(len(uniq) + 1))
    return [order[bounds[i]:bounds[i + 1]] for i in range(len(uniq))]


def bootstrap(stat_fn, arrays, groups, n_boot=1000, seed=0):
    """Cluster bootstrap (resampling `groups`, e.g. patients).

    stat_fn(*arrays_subset) -> dict of floats. Returns {key: (lo, hi)} 95% CIs.
    """
    rng = np.random.default_rng(seed)
    idx = _group_index(groups)
    samples = {}
    for _ in range(n_boot):
        pick = rng.integers(0, len(idx), len(idx))
        sel = np.concatenate([idx[i] for i in pick])
        for k, v in stat_fn(*[a[sel] for a in arrays]).items():
            samples.setdefault(k, []).append(v)
    return {k: (float(np.nanpercentile(v, 2.5)), float(np.nanpercentile(v, 97.5))) for k, v in samples.items()}


def paired_bootstrap(stat_fn, y, p_a, p_b, groups, n_boot=1000, seed=0):
    """Difference stat(a) - stat(b) with 95% CI and two-sided bootstrap p-value."""
    rng = np.random.default_rng(seed)
    idx = _group_index(groups)
    obs = stat_fn(y, p_a) - stat_fn(y, p_b)
    diffs = []
    for _ in range(n_boot):
        sel = np.concatenate([idx[i] for i in rng.integers(0, len(idx), len(idx))])
        diffs.append(stat_fn(y[sel], p_a[sel]) - stat_fn(y[sel], p_b[sel]))
    diffs = np.asarray(diffs)
    p = 2 * min((diffs <= 0).mean(), (diffs >= 0).mean())
    return {"diff": float(obs), "ci": (float(np.percentile(diffs, 2.5)), float(np.percentile(diffs, 97.5))),
            "p_value": float(min(p, 1.0))}


In [ ]:
%%writefile {CODE}/scripts/audit.py
"""Dataset audit for ReXErr-v1.

Produces results/audit.json with the statistics reported in the revised
manuscript: split sizes, patient overlap, label-source agreement, label
prevalence, report lengths, whitespace shortcut, and the leakage that a random
report-level split would introduce.

usage: python scripts/audit.py --data_dir DATA --out results/audit.json [--tokenizer dmis-lab/biobert-v1.1]
"""
import argparse
import json
import os
import sys

import numpy as np
import pandas as pd

sys.path.insert(0, os.path.join(os.path.dirname(__file__), ".."))
from rexerr.data import SPLITS, load_reports, load_sentences, normalize_text, sentence_label_sets  # noqa: E402
from rexerr.labels import NATIVE, GROUP6, group6_matrix, parse_sampled  # noqa: E402
from rexerr.data import to_multihot  # noqa: E402


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data_dir", required=True)
    ap.add_argument("--out", default="results/audit.json")
    ap.add_argument("--tokenizer", default=None, help="HF tokenizer for token-length stats")
    args = ap.parse_args()

    out = {"splits": {}, "labels": {}}
    reps = {s: load_reports(args.data_dir, s) for s in SPLITS}
    sents = {s: load_sentences(args.data_dir, s) for s in SPLITS}

    # Split sizes and patient overlap.
    for s in SPLITS:
        r = reps[s]
        out["splits"][s] = {
            "reports": int(len(r)), "subjects": int(r["subject_id"].nunique()),
            "unique_study_ids": int(r["study_id"].nunique()),
            "sentences": int(len(sents[s])),
            "studies_with_sentence_labels": int(sents[s]["study_id"].nunique()),
        }
    subj = {s: set(reps[s]["subject_id"]) for s in SPLITS}
    out["patient_overlap"] = {
        "train_test": len(subj["train"] & subj["test"]), "train_val": len(subj["train"] & subj["val"]),
        "val_test": len(subj["val"] & subj["test"]),
    }

    # Exact duplicate report texts across splits (templated normal reports).
    norm = {s: set(reps[s]["original_report"].map(normalize_text)) for s in SPLITS}
    test_norm = reps["test"]["original_report"].map(normalize_text)
    out["duplicates"] = {
        "test_original_reports_with_identical_text_in_train": int(test_norm.isin(norm["train"]).sum()),
        "test_reports": int(len(test_norm)),
    }
    out["unchanged_error_reports"] = {
        s: int((reps[s]["error_report"].map(normalize_text) == reps[s]["original_report"].map(normalize_text)).sum())
        for s in SPLITS
    }

    # Whitespace / formatting shortcut.
    def frac(series, pat):
        return float(series.fillna("").str.contains(pat, regex=True).mean())
    r = reps["train"]
    out["formatting_shortcut_train"] = {
        "original_with_double_space": frac(r["original_report"], r"  "),
        "error_with_double_space": frac(r["error_report"], r"  "),
        "original_with_trailing_space": frac(r["original_report"], r"\s$"),
        "error_with_trailing_space": frac(r["error_report"], r"\s$"),
        "original_with_newline": frac(r["original_report"], r"\n"),
        "error_with_newline": frac(r["error_report"], r"\n"),
    }

    # Lengths.
    lens = {}
    for v in ("original_report", "error_report"):
        t = r[v].map(normalize_text)
        w = t.str.split().str.len()
        lens[v] = {"chars_mean": float(t.str.len().mean()), "chars_median": float(t.str.len().median()),
                   "words_mean": float(w.mean()), "words_median": float(w.median()),
                   "words_p05": float(w.quantile(0.05)), "frac_under_50_words": float((w < 50).mean())}
    out["lengths_train"] = lens
    if args.tokenizer:
        from transformers import AutoTokenizer
        tok = AutoTokenizer.from_pretrained(args.tokenizer)
        sample = r["error_report"].map(normalize_text).sample(min(20000, len(r)), random_state=0).tolist()
        n = np.array([len(x) for x in tok(sample, add_special_tokens=True)["input_ids"]])
        out["tokens_error_report_train"] = {
            "tokenizer": args.tokenizer, "mean": float(n.mean()), "median": float(np.median(n)),
            "p95": float(np.percentile(n, 95)), "p99": float(np.percentile(n, 99)),
            "max": int(n.max()), "frac_over_256": float((n > 256).mean()), "frac_over_512": float((n > 512).mean()),
        }

    # Label sources: errors_sampled vs sentence-level annotations.
    for s in SPLITS:
        rep = reps[s]
        sampled = [set(parse_sampled(v)) for v in rep["errors_sampled"]]
        sent_sets = sentence_label_sets(sents[s])
        sent = [sent_sets.get(x, set()) for x in rep["study_id"]]
        ys, yt = to_multihot(sampled), to_multihot(sent)
        per = {}
        for j, k in enumerate(NATIVE):
            a, b = ys[:, j] == 1, yt[:, j] == 1
            per[k] = {
                "prevalence_sampled": float(a.mean()), "prevalence_sentence": float(b.mean()),
                "sampled_confirmed_by_sentence": float((a & b).sum() / a.sum()) if a.sum() else None,
                "sentence_also_sampled": float((a & b).sum() / b.sum()) if b.sum() else None,
            }
        g_s, g_t = group6_matrix(ys), group6_matrix(yt)
        out["labels"][s] = {
            "per_type": per,
            "mean_types_per_report_sampled": float(ys.sum(1).mean()),
            "mean_types_per_report_sentence": float(yt.sum(1).mean()),
            "exact_agreement_rate": float((ys == yt).all(1).mean()),
            "reports_with_no_sentence_level_error": float((yt.sum(1) == 0).mean()),
            "group6_prevalence_sampled": dict(zip(GROUP6, g_s.mean(0).round(4).tolist())),
            "group6_prevalence_sentence": dict(zip(GROUP6, g_t.mean(0).round(4).tolist())),
            "group6_mean_per_report_sampled": float(g_s.sum(1).mean()),
            "error_present_counts": {str(k): int(v) for k, v in sents[s]["error_present"].value_counts().items()},
        }

    # Leakage of a random 70/15/15 report-level split.
    allrep = pd.concat([reps[s] for s in SPLITS], ignore_index=True)
    rng = np.random.default_rng(0)
    perm = rng.permutation(len(allrep))
    n_tr = int(0.70 * len(allrep)); n_va = int(0.15 * len(allrep))
    tr, te = allrep.iloc[perm[:n_tr]], allrep.iloc[perm[n_tr + n_va:]]
    out["random_split_leakage"] = {
        "test_reports_whose_patient_is_in_train": float(te["subject_id"].isin(set(tr["subject_id"])).mean()),
        "note": "70/15/15 random report-level split, seed 0",
    }

    # Clinician plausibility review shipped with the dataset.
    cr_path = os.path.join(args.data_dir, "clinician-review.csv")
    if os.path.exists(cr_path):
        cr = pd.read_csv(cr_path)
        out["clinician_review"] = {"n": int(len(cr)),
                                   "acceptable_rate": float(cr["acceptable"].astype(str).str.lower().str.startswith("y").mean())}

    os.makedirs(os.path.dirname(os.path.abspath(args.out)), exist_ok=True)
    with open(args.out, "w") as f:
        json.dump(out, f, indent=2)
    print(json.dumps(out, indent=2))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile {CODE}/scripts/baselines.py
"""Non-neural baselines (CPU): report-length logistic regression and TF-IDF + logistic regression.

usage: python scripts/baselines.py --data_dir DATA --runs_dir results/runs --model tfidf|length
"""
import argparse
import os
import sys
import time

import numpy as np
from scipy.sparse import hstack
from sklearn.feature_extraction.text import HashingVectorizer, TfidfTransformer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import StandardScaler

sys.path.insert(0, os.path.join(os.path.dirname(__file__), ".."))
from rexerr.data import build_report_task  # noqa: E402
from rexerr.io import save_predictions  # noqa: E402


def length_features(texts):
    t = np.array(texts, dtype=object)
    chars = np.array([len(x) for x in t], dtype=float)
    words = np.array([len(x.split()) for x in t], dtype=float)
    sents = np.array([max(1, x.count(". ") + 1) for x in t], dtype=float)
    return np.log1p(np.stack([chars, words, sents], axis=1))


def fit_predict(X_tr, Y, X_eval, C):
    active = Y.sum(0) > 0
    # liblinear is not thread-safe, so the one-vs-rest fits run sequentially
    clf = OneVsRestClassifier(LogisticRegression(solver="liblinear", C=C, max_iter=1000))
    clf.fit(X_tr, Y[:, active])
    outs = []
    for X in X_eval:
        p = np.zeros((X.shape[0], Y.shape[1]), dtype=np.float32)
        p[:, active] = clf.predict_proba(X)
        outs.append(p)
    return outs


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data_dir", required=True)
    ap.add_argument("--runs_dir", default="results/runs")
    ap.add_argument("--model", choices=["tfidf", "length"], default="tfidf")
    ap.add_argument("--label_source", default="sentence", choices=["sentence", "sampled"])
    ap.add_argument("--include_clean", type=int, default=1)
    ap.add_argument("--max_train_studies", type=int, default=None)
    ap.add_argument("--C", type=float, default=4.0)
    ap.add_argument("--max_features", type=int, default=300000)
    ap.add_argument("--tag", default="")
    args = ap.parse_args()

    t0 = time.time()
    tr, Ytr = build_report_task(args.data_dir, "train", args.label_source, bool(args.include_clean),
                                max_studies=args.max_train_studies)
    va, Yva = build_report_task(args.data_dir, "val", args.label_source, True)
    te, Yte = build_report_task(args.data_dir, "test", args.label_source, True)
    print(f"train {len(tr)} val {len(va)} test {len(te)} ({time.time() - t0:.0f}s)", flush=True)

    if args.model == "length":
        sc = StandardScaler().fit(length_features(tr.text))
        X_tr, X_va, X_te = (sc.transform(length_features(d.text)) for d in (tr, va, te))
        C = 1.0
    else:
        word = TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_features=args.max_features,
                               sublinear_tf=True, dtype=np.float32, lowercase=False)
        # Character n-grams are hashed: an explicit vocabulary for 400k reports does not fit in 16 GB.
        hasher = HashingVectorizer(analyzer="char_wb", ngram_range=(3, 5), n_features=2 ** 20, alternate_sign=False,
                                   norm=None, dtype=np.float32, lowercase=False)
        char = TfidfTransformer(sublinear_tf=True)
        X_tr = hstack([word.fit_transform(tr.text), char.fit_transform(hasher.transform(tr.text))], format="csr",
                      dtype=np.float32)
        X_va = hstack([word.transform(va.text), char.transform(hasher.transform(va.text))], format="csr", dtype=np.float32)
        X_te = hstack([word.transform(te.text), char.transform(hasher.transform(te.text))], format="csr", dtype=np.float32)
        print(f"features {X_tr.shape[1]} nnz {X_tr.nnz} ({time.time() - t0:.0f}s)", flush=True)
        C = args.C

    p_va, p_te = fit_predict(X_tr, Ytr, [X_va, X_te], C)
    name = f"{args.model}_{args.label_source}_clean{args.include_clean}{args.tag}"
    out_dir = os.path.join(args.runs_dir, name)
    meta = {"model": args.model, "family": "baseline", "task": "report", "seed": 0,
            "label_source": args.label_source, "include_clean": args.include_clean,
            "n_train": len(tr), "runtime_sec": time.time() - t0, "args": vars(args)}
    save_predictions(out_dir, "val", va, Yva, p_va)
    save_predictions(out_dir, "test", te, Yte, p_te, meta)
    print(f"saved {out_dir} ({time.time() - t0:.0f}s)")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile {CODE}/scripts/train_encoder.py
"""Fine-tune a transformer encoder for multi-label error detection (GPU).

Report task: input = report text, output = 12 native ReXErr error types.
Sentence task (localization): input = (sentence, full report), output = the
error type of that sentence (all zeros if the sentence is correct).

Checkpoints are written to --out_dir and training resumes automatically, so a
disconnected Colab/Kaggle session can simply be re-run.

usage: python scripts/train_encoder.py --data_dir DATA --model dmis-lab/biobert-v1.1 --seed 0 \
           --out_dir results/runs/biobert_s0
"""
import argparse
import glob
import inspect
import os
import sys
import time

import numpy as np
import torch
from torch import nn
from transformers import (AutoModel, AutoTokenizer, BertModel, BertTokenizerFast, EarlyStoppingCallback, Trainer,
                          TrainingArguments, set_seed)

sys.path.insert(0, os.path.join(os.path.dirname(__file__), ".."))
from rexerr.data import build_report_task, build_sentence_task  # noqa: E402
from rexerr.io import save_predictions  # noqa: E402
from rexerr.labels import NATIVE  # noqa: E402
from rexerr.metrics import summary  # noqa: E402


def load_encoder(name):
    try:
        model = AutoModel.from_pretrained(name)
    except ValueError:  # e.g. BlueBERT, whose config.json has no model_type
        model = BertModel.from_pretrained(name)
    for p in model.parameters():  # weights converted from TensorFlow can be non-contiguous, which safetensors rejects
        p.data = p.data.contiguous()
    return model


def load_tokenizer(name):
    try:
        return AutoTokenizer.from_pretrained(name)
    except ValueError:
        return BertTokenizerFast.from_pretrained(name, do_lower_case="uncased" in name.lower())


class MultiLabelEncoder(nn.Module):
    """Encoder + classification head on the [CLS] representation.

    loss: "bce" (unweighted), "weighted" (BCE with per-type positive weights = negatives / positives, capped at
    100) or "focal" (binary focal loss).
    """

    def __init__(self, name, n_labels, head="mlp", dropout=0.1, loss="bce", pos_weight=None, focal_gamma=2.0):
        super().__init__()
        self.encoder = load_encoder(name)
        h = self.encoder.config.hidden_size
        if head == "mlp":  # 768 -> 256 -> ReLU -> labels
            self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(h, 256), nn.ReLU(), nn.Dropout(dropout),
                                      nn.Linear(256, n_labels))
        else:
            self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(h, n_labels))
        self.loss, self.focal_gamma = loss, focal_gamma
        self.register_buffer("pos_weight", torch.as_tensor(pos_weight if pos_weight is not None else np.ones(n_labels),
                                                           dtype=torch.float32))

    def compute_loss(self, logits, labels):
        if self.loss == "weighted":
            return nn.functional.binary_cross_entropy_with_logits(logits, labels, pos_weight=self.pos_weight)
        bce = nn.functional.binary_cross_entropy_with_logits(logits, labels, reduction="none")
        if self.loss == "focal":
            p_t = torch.exp(-bce)
            return ((1 - p_t) ** self.focal_gamma * bce).mean()
        return bce.mean()

    def forward(self, input_ids, attention_mask, token_type_ids=None, labels=None):
        kw = {"input_ids": input_ids, "attention_mask": attention_mask}
        if token_type_ids is not None:
            kw["token_type_ids"] = token_type_ids
        logits = self.head(self.encoder(**kw).last_hidden_state[:, 0])
        out = {"logits": logits}
        if labels is not None:
            out["loss"] = self.compute_loss(logits.float(), labels.float())
        return out


class EncodedDataset(torch.utils.data.Dataset):
    def __init__(self, enc, labels):
        self.ids = [np.asarray(x, dtype=np.int32) for x in enc["input_ids"]]
        self.tt = [np.asarray(x, dtype=np.int8) for x in enc["token_type_ids"]] if "token_type_ids" in enc else None
        self.labels = labels

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, i):
        item = {"input_ids": self.ids[i], "labels": self.labels[i]}
        if self.tt is not None:
            item["token_type_ids"] = self.tt[i]
        return item


class Collator:
    def __init__(self, pad_id):
        self.pad_id = pad_id

    def __call__(self, batch):
        n = max(len(b["input_ids"]) for b in batch)
        ids = torch.full((len(batch), n), self.pad_id, dtype=torch.long)
        mask = torch.zeros((len(batch), n), dtype=torch.long)
        tt = torch.zeros((len(batch), n), dtype=torch.long) if "token_type_ids" in batch[0] else None
        for i, b in enumerate(batch):
            k = len(b["input_ids"])
            ids[i, :k] = torch.from_numpy(b["input_ids"].astype(np.int64))
            mask[i, :k] = 1
            if tt is not None:
                tt[i, :k] = torch.from_numpy(b["token_type_ids"].astype(np.int64))
        out = {"input_ids": ids, "attention_mask": mask,
               "labels": torch.tensor(np.stack([b["labels"] for b in batch]), dtype=torch.float32)}
        if tt is not None:
            out["token_type_ids"] = tt
        return out


def encode(tok, frame, task, max_len, lowercase):
    text = frame["text"].tolist()
    if lowercase:
        text = [t.lower() for t in text]
    if task == "sentence":
        ctx = frame["context"].tolist()
        if lowercase:
            ctx = [c.lower() for c in ctx]
        return tok(text, ctx, truncation="only_second", max_length=max_len)
    return tok(text, truncation=True, max_length=max_len)


def sigmoid(x):
    return 1 / (1 + np.exp(-x))


def versions():
    import platform

    import sklearn
    import tokenizers
    import transformers
    return {"python": platform.python_version(), "torch": torch.__version__, "cuda": torch.version.cuda,
            "transformers": transformers.__version__, "tokenizers": tokenizers.__version__,
            "scikit-learn": sklearn.__version__, "numpy": np.__version__}


def training_arguments(**kw):
    """TrainingArguments across transformers 4.x and 5.x (renamed/removed options)."""
    params = inspect.signature(TrainingArguments.__init__).parameters
    if "group_by_length" in params:
        kw["group_by_length"] = True
    elif "train_sampling_strategy" in params:
        kw["train_sampling_strategy"] = "group_by_length"
    if "eval_strategy" not in params:
        kw["evaluation_strategy"] = kw.pop("eval_strategy")
    return TrainingArguments(**{k: v for k, v in kw.items() if k in params})


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data_dir", required=True)
    ap.add_argument("--out_dir", required=True)
    ap.add_argument("--model", default="dmis-lab/biobert-v1.1")
    ap.add_argument("--task", choices=["report", "sentence"], default="report")
    ap.add_argument("--label_source", choices=["sentence", "sampled"], default="sentence")
    ap.add_argument("--include_clean", type=int, default=1, help="add original reports as negatives to TRAINING")
    ap.add_argument("--max_train_studies", type=int, default=None)
    ap.add_argument("--max_eval_studies", type=int, default=None)
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--epochs", type=float, default=2.0)
    ap.add_argument("--lr", type=float, default=2e-5)
    ap.add_argument("--batch_size", type=int, default=32)
    ap.add_argument("--grad_accum", type=int, default=1)
    ap.add_argument("--max_len", type=int, default=320)
    ap.add_argument("--head", choices=["mlp", "linear"], default="mlp")
    ap.add_argument("--loss", choices=["bce", "weighted", "focal"], default="bce")
    ap.add_argument("--focal_gamma", type=float, default=2.0)
    ap.add_argument("--save_model", type=int, default=1, help="keep the best weights (fp16) for later re-scoring")
    ap.add_argument("--lowercase", type=int, default=0)
    ap.add_argument("--evals_per_epoch", type=int, default=4)
    ap.add_argument("--patience", type=int, default=3)
    ap.add_argument("--workers", type=int, default=2)
    ap.add_argument("--max_steps", type=int, default=-1, help="debug only")
    args = ap.parse_args()

    t0 = time.time()
    set_seed(args.seed)
    # bf16 on Ampere or newer GPUs (L4, A100, H100, G4); fp16 on T4
    # is_bf16_supported() is also True on a T4, where bf16 is emulated and several times slower than fp16
    use_bf16 = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8
    if torch.cuda.is_available():
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
    builder = build_report_task if args.task == "report" else build_sentence_task
    kw = {} if args.task == "sentence" else {"label_source": args.label_source}
    tr, Ytr = builder(args.data_dir, "train", include_clean=bool(args.include_clean),
                      max_studies=args.max_train_studies, seed=args.seed, **kw)
    # Evaluation always includes the error-free originals, whatever was used for training.
    va, Yva = builder(args.data_dir, "val", include_clean=True, max_studies=args.max_eval_studies, **kw)
    te, Yte = builder(args.data_dir, "test", include_clean=True, max_studies=args.max_eval_studies, **kw)
    print(f"[data] train {len(tr)} val {len(va)} test {len(te)} ({time.time() - t0:.0f}s)", flush=True)

    tok = load_tokenizer(args.model)
    ds = {name: EncodedDataset(encode(tok, f, args.task, args.max_len, args.lowercase), y)
          for name, f, y in (("train", tr, Ytr), ("val", va, Yva), ("test", te, Yte))}
    trunc = float(np.mean([len(x) >= args.max_len for x in ds["train"].ids]))
    print(f"[tok] truncated at {args.max_len}: {trunc:.4%} ({time.time() - t0:.0f}s)", flush=True)

    pos, neg = Ytr.sum(0), len(Ytr) - Ytr.sum(0)
    pos_weight = np.where(pos > 0, np.clip(neg / np.maximum(pos, 1), 1.0, 100.0), 1.0)
    model = MultiLabelEncoder(args.model, len(NATIVE), head=args.head, loss=args.loss,
                              pos_weight=pos_weight if args.loss == "weighted" else None, focal_gamma=args.focal_gamma)
    n_params = sum(p.numel() for p in model.parameters())
    n_dev = max(1, torch.cuda.device_count())
    steps_per_epoch = max(1, len(ds["train"]) // (args.batch_size * args.grad_accum * n_dev))
    eval_steps = max(1, steps_per_epoch // args.evals_per_epoch)
    active = Ytr.sum(0) > 0

    def compute_metrics(pred):
        p = sigmoid(pred.predictions)
        return summary(pred.label_ids, p, 0.5, mask=active & (pred.label_ids.sum(0) > 0))

    total_steps = args.max_steps if args.max_steps > 0 else int(steps_per_epoch * args.epochs)
    targs = training_arguments(
        output_dir=args.out_dir, seed=args.seed, data_seed=args.seed,
        num_train_epochs=args.epochs, max_steps=args.max_steps, learning_rate=args.lr,
        per_device_train_batch_size=args.batch_size, per_device_eval_batch_size=args.batch_size * 2,
        gradient_accumulation_steps=args.grad_accum, warmup_steps=max(1, int(0.1 * total_steps)),
        weight_decay=0.01, lr_scheduler_type="linear", max_grad_norm=1.0,
        fp16=torch.cuda.is_available() and not use_bf16, bf16=use_bf16,
        eval_strategy="steps", eval_steps=eval_steps, save_strategy="steps", save_steps=eval_steps,
        save_total_limit=2, load_best_model_at_end=True, metric_for_best_model="f1_macro",
        greater_is_better=True, logging_steps=max(1, eval_steps // 4), report_to="none",
        dataloader_num_workers=args.workers, remove_unused_columns=False, disable_tqdm=True,
    )
    trainer = Trainer(model=model, args=targs, train_dataset=ds["train"], eval_dataset=ds["val"],
                      data_collator=Collator(tok.pad_token_id), compute_metrics=compute_metrics,
                      callbacks=[EarlyStoppingCallback(early_stopping_patience=args.patience)])
    resume = bool(glob.glob(os.path.join(args.out_dir, "checkpoint-*")))
    train_out = trainer.train(resume_from_checkpoint=True if resume else None)

    preds = {}
    for name, frame, y in (("val", va, Yva), ("test", te, Yte)):
        logits = trainer.predict(ds[name]).predictions
        preds[name] = sigmoid(logits)
    history = [h for h in trainer.state.log_history if "eval_f1_macro" in h or "loss" in h]
    if args.save_model:
        from safetensors.torch import save_file
        state = {k: v.detach().to("cpu", torch.float16).contiguous() for k, v in trainer.model.state_dict().items()}
        save_file(state, os.path.join(args.out_dir, "model_fp16.safetensors"))
    meta = {
        "model": args.model, "family": "encoder", "task": args.task, "seed": args.seed,
        "label_source": args.label_source, "include_clean": args.include_clean,
        "n_train": len(tr), "n_params": n_params, "truncated_frac": trunc,
        "best_checkpoint": trainer.state.best_model_checkpoint, "best_val_f1_macro": trainer.state.best_metric,
        "train_runtime_sec": train_out.metrics.get("train_runtime"), "total_runtime_sec": time.time() - t0,
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
        "precision": "bf16" if use_bf16 else ("fp16" if torch.cuda.is_available() else "fp32"),
        "global_steps": trainer.state.global_step, "loss": args.loss,
        "pos_weight": pos_weight.tolist() if args.loss == "weighted" else None,
        "versions": versions(), "resumed": resume, "args": vars(args), "history": history,
    }
    save_predictions(args.out_dir, "val", va, Yva, preds["val"])
    save_predictions(args.out_dir, "test", te, Yte, preds["test"], meta)
    for ck in glob.glob(os.path.join(args.out_dir, "checkpoint-*")):  # keep Drive small
        import shutil
        shutil.rmtree(ck, ignore_errors=True)
    print(f"[done] {args.out_dir} best val f1_macro={trainer.state.best_metric} ({time.time() - t0:.0f}s)")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile {CODE}/scripts/llm_zeroshot.py
"""Zero-shot / few-shot LLM baseline on a patient-level random subset of the test split (GPU).

The LLM runs locally (4-bit), so no report text leaves the machine.
Outputs are binary (type flagged or not), stored in the common prediction format.

usage: python scripts/llm_zeroshot.py --data_dir DATA --out_dir results/runs/llm_qwen2.5-7b \
           --model Qwen/Qwen2.5-7B-Instruct --n_studies 500
"""
import argparse
import json
import os
import re
import sys
import time

import numpy as np
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

sys.path.insert(0, os.path.join(os.path.dirname(__file__), ".."))
from rexerr.data import build_report_task  # noqa: E402
from rexerr.io import save_predictions  # noqa: E402
from rexerr.labels import NATIVE, NATIVE_DISPLAY, normalize_type  # noqa: E402

DEFINITIONS = {
    "add_medical_device": "a medical device (line, tube, catheter, pacemaker...) is mentioned that is unlikely to be present",
    "false_prediction": "a finding is reported that is unlikely to be present",
    "false_negation": "a finding is wrongly negated (stated as absent)",
    "add_repetitions": "a statement is repeated unnecessarily",
    "add_contradictions": "two statements in the report contradict each other",
    "change_name_of_device": "a device is named incorrectly",
    "change_position_of_device": "the position of a device is described incorrectly",
    "change_severity": "the severity/size of a finding is misstated",
    "change_location": "the anatomical location or laterality of a finding is wrong",
    "change_measurement": "a measurement or number is wrong",
    "change_to_homophone": "a word is replaced by a similar-sounding wrong word (speech-recognition error)",
    "add_typo": "a typographical error",
}

SYSTEM = "You are an experienced radiologist who proofreads chest X-ray reports for errors."


def prompt(report):
    lines = "\n".join(f'- "{NATIVE_DISPLAY[k]}": {v}' for k, v in DEFINITIONS.items())
    return (
        "The following chest X-ray report may contain errors of these types:\n"
        f"{lines}\n\n"
        "Many reports contain no error at all. Read the report and list every error type that is present.\n"
        'Answer with JSON only, for example {"errors": ["Add typo", "Change severity"]} or {"errors": []}.\n\n'
        f"Report:\n{report}"
    )


def parse(text):
    names = {NATIVE_DISPLAY[k].lower(): k for k in NATIVE}
    found = set()
    text = re.sub(r"<think>.*?(</think>|$)", "", text, flags=re.S)  # reasoning models (thinking is disabled anyway)
    m = re.search(r"\{.*\}", text, flags=re.S)
    if m:
        try:
            for e in json.loads(m.group(0)).get("errors", []):
                k = normalize_type(e)
                if k:
                    found.add(k)
            return found, True
        except (json.JSONDecodeError, AttributeError):
            pass
    low = text.lower()
    return {k for n, k in names.items() if n in low}, False


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data_dir", required=True)
    ap.add_argument("--out_dir", required=True)
    ap.add_argument("--model", default="Qwen/Qwen2.5-7B-Instruct")
    ap.add_argument("--n_studies", type=int, default=500, help="0 = the whole test split")
    ap.add_argument("--batch_size", type=int, default=8)
    ap.add_argument("--max_new_tokens", type=int, default=80)
    ap.add_argument("--load_in_4bit", type=int, default=-1, help="-1 = only when all GPUs together have < 20 GB")
    ap.add_argument("--seed", type=int, default=0)
    args = ap.parse_args()

    t0 = time.time()
    te, Yte = build_report_task(args.data_dir, "test", "sentence", True, max_studies=args.n_studies or None, seed=args.seed)
    va, Yva = te.iloc[:0], Yte[:0]
    tok = AutoTokenizer.from_pretrained(args.model)
    tok.padding_side = "left"
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    cuda = torch.cuda.is_available()
    native_bf16 = cuda and torch.cuda.get_device_capability(0)[0] >= 8  # a T4 only emulates bf16
    dtype = (torch.bfloat16 if native_bf16 else torch.float16) if cuda else torch.float32
    if args.load_in_4bit == -1:
        total = sum(torch.cuda.get_device_properties(i).total_memory for i in range(torch.cuda.device_count()))
        args.load_in_4bit = int(cuda and total < 20e9)  # device_map="auto" spreads the model over all GPUs
    kw = {"dtype": dtype, "device_map": "auto"}
    if args.load_in_4bit:
        from transformers import BitsAndBytesConfig
        kw["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=dtype,
                                                       bnb_4bit_quant_type="nf4")
    model = AutoModelForCausalLM.from_pretrained(args.model, **kw).eval()

    texts = [tok.apply_chat_template([{"role": "system", "content": SYSTEM}, {"role": "user", "content": prompt(r)}],
                                     tokenize=False, add_generation_prompt=True,
                                     enable_thinking=False)  # Qwen3: answer directly; ignored by other templates
             for r in te["text"]]
    P = np.zeros((len(te), len(NATIVE)), dtype=np.float32)
    raw, parsed_ok = [], 0
    for i in range(0, len(texts), args.batch_size):
        enc = tok(texts[i:i + args.batch_size], return_tensors="pt", padding=True).to(model.device)
        with torch.no_grad():
            out = model.generate(**enc, max_new_tokens=args.max_new_tokens, do_sample=False,
                                 pad_token_id=tok.pad_token_id)
        for j, seq in enumerate(out[:, enc["input_ids"].shape[1]:]):
            ans = tok.decode(seq, skip_special_tokens=True)
            found, ok = parse(ans)
            parsed_ok += ok
            raw.append(ans)
            for k in found:
                P[i + j, NATIVE.index(k)] = 1.0
        print(f"[llm] {min(i + args.batch_size, len(texts))}/{len(texts)} ({time.time() - t0:.0f}s)", flush=True)

    meta = {"model": args.model, "family": "llm_zeroshot", "task": "report", "seed": args.seed,
            "label_source": "sentence", "include_clean": 1, "n_test": len(te), "json_parse_rate": parsed_ok / len(te),
            "runtime_sec": time.time() - t0, "args": vars(args),
            "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"}
    save_predictions(args.out_dir, "val", va, Yva, np.zeros((0, len(NATIVE))))
    save_predictions(args.out_dir, "test", te, Yte, P, meta)
    with open(os.path.join(args.out_dir, "raw_outputs.jsonl"), "w") as f:
        for u, a in zip(te["uid"], raw):
            f.write(json.dumps({"uid": u, "output": a}) + "\n")
    print(f"[done] {args.out_dir} parse rate {parsed_ok / len(te):.3f} ({time.time() - t0:.0f}s)")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile {CODE}/scripts/evaluate.py
"""Evaluate every run in --runs_dir with one code path and write the paper's tables.

Outputs (in --out_dir):
  summary.json        all metrics, CIs, seed aggregates, paired comparisons, prevalence analysis
  tables.md           human-readable tables used in the manuscript
  error_examples.csv  qualitative false positives / negatives of the main model (needs --data_dir)

usage: python scripts/evaluate.py --runs_dir results/runs --out_dir results --main biobert [--data_dir DATA]
"""
import argparse
import glob
import json
import os
import sys
from collections import defaultdict

import numpy as np

sys.path.insert(0, os.path.join(os.path.dirname(__file__), ".."))
from rexerr import metrics as M  # noqa: E402
from rexerr.io import load_meta, load_predictions  # noqa: E402
from rexerr.labels import GROUP6, GROUP6_DISPLAY, NATIVE, NATIVE_DISPLAY, group6_matrix  # noqa: E402

DISPLAY = {
    "dmis-lab/biobert-v1.1": "BioBERT v1.1",
    "bert-base-cased": "BERT-base (cased)",
    "bert-base-uncased": "BERT-base (uncased)",
    "emilyalsentzer/Bio_ClinicalBERT": "Bio+ClinicalBERT",
    "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext": "BiomedBERT (PubMedBERT)",
    "StanfordAIMI/RadBERT": "RadBERT",
    "microsoft/BiomedVLP-CXR-BERT-general": "CXR-BERT",
    "UCSD-VA-health/RadBERT-RoBERTa-4m": "RadBERT (RoBERTa)",
    "bionlp/bluebert_pubmed_mimic_uncased_L-12_H-768_A-12": "BlueBERT",
    "unsloth/Meta-Llama-3.1-8B-Instruct": "Llama-3.1-8B-Instruct (zero-shot)",
    "tfidf": "TF-IDF + LR",
    "length": "Report length + LR",
    "Qwen/Qwen2.5-7B-Instruct": "Qwen2.5-7B-Instruct (zero-shot)",
    "Qwen/Qwen3-8B": "Qwen3-8B (zero-shot)",
}
HEADLINE = ["f1_macro", "f1_micro", "auroc_macro", "auprc_macro", "any_auroc", "any_auprc"]


def display(meta):
    name = DISPLAY.get(meta.get("model"), meta.get("model", "?"))
    if meta.get("task") == "sentence":
        name += " [sentence-level]"
    if str(meta.get("include_clean", 1)) == "0":
        name += " [trained without clean reports]"
    if meta.get("label_source") == "sampled":
        name += " [errors_sampled labels]"
    if meta.get("loss") in ("weighted", "focal"):
        name += " [weighted BCE]" if meta["loss"] == "weighted" else " [focal loss]"
    tag = (meta.get("args") or {}).get("tag")
    if tag:
        name += f" [{tag.strip('_')} training studies]" if tag.strip("_").endswith("k") else f" [{tag.strip('_')}]"
    return name


def aggregate_sentences(d):
    """Sentence predictions -> report predictions (max over the report's sentences)."""
    keys = np.char.add(d["study_id"].astype(str), np.char.add("_", d["variant"]))
    uniq, inv = np.unique(keys, return_inverse=True)
    y = np.zeros((len(uniq), d["y"].shape[1]), dtype=np.int8)
    p = np.zeros((len(uniq), d["p"].shape[1]), dtype=np.float32)
    np.maximum.at(y, inv, d["y"])
    np.maximum.at(p, inv, d["p"])
    first = np.zeros(len(uniq), dtype=int)
    first[inv[::-1]] = np.arange(len(inv))[::-1]
    return {"uid": uniq, "study_id": d["study_id"][first], "subject_id": d["subject_id"][first],
            "variant": d["variant"][first], "y": y, "p": p}


def headline_fn(mask):
    def fn(y, p):
        s = M.summary(y, p, 0.5, mask=mask)
        b = M.binary(y.max(1), M.any_error_score(p))
        return {"f1_macro": s["f1_macro"], "f1_micro": s["f1_micro"], "auroc_macro": s["auroc_macro"],
                "auprc_macro": s["auprc_macro"], "any_auroc": b["auroc"], "any_auprc": b["auprc"]}
    return fn


def evaluate_report_level(val, test, n_boot, seed=0):
    y, p, yv, pv = test["y"].astype(int), test["p"], val["y"].astype(int), val["p"]
    mask = M.active_classes(y)
    res = {"n_test": int(len(y)), "n_val": int(len(yv)), "active_classes": [c for c, m in zip(NATIVE, mask) if m]}
    res["native"] = M.summary(y, p, 0.5, mask=mask)
    res["native_per_class"] = M.per_class(y, p, NATIVE, 0.5)
    thr = M.tune_thresholds(yv, pv)
    res["native_tuned"] = M.summary(y, p, thr, mask=mask)
    res["native_tuned_per_class"] = M.per_class(y, p, NATIVE, thr)
    y6, p6 = group6_matrix(y), group6_matrix(p)
    res["group6"] = M.summary(y6, p6, 0.5)
    res["group6_per_class"] = M.per_class(y6, p6, GROUP6, 0.5)

    y_any, s_any = y.max(1), M.any_error_score(p)
    yv_any, sv_any = yv.max(1), M.any_error_score(pv)
    res["any_error"] = {"at_0.5": M.binary(y_any, s_any, 0.5)}
    for target in (0.90, 0.95):
        t = M.threshold_for_sensitivity(yv_any, sv_any, target)
        b = M.binary(y_any, s_any, t)
        res["any_error"][f"val_sens_{target:.2f}"] = b
        res["any_error"][f"prevalence_sens_{target:.2f}"] = M.prevalence_analysis(b["sensitivity"], b["specificity"])
    b = res["any_error"]["at_0.5"]
    res["any_error"]["prevalence_at_0.5"] = M.prevalence_analysis(b["sensitivity"], b["specificity"])

    err = test["variant"] == "error"
    if err.any() and (~err).any():  # error reports only = evaluation without controls
        res["error_reports_only"] = M.summary(y[err], p[err], 0.5, mask=M.active_classes(y[err]))

    res["ci95"] = M.bootstrap(headline_fn(mask), [y, p], test["subject_id"], n_boot=n_boot, seed=seed)
    return res


def evaluate_sentence_level(test):
    y, p = test["y"].astype(int), test["p"]
    mask = M.active_classes(y)
    err_rows = test["variant"] == "error"
    return {
        "n_sentences": int(len(y)),
        "localization": M.binary(y.max(1), M.any_error_score(p)),
        "localization_error_reports_only": M.binary(y[err_rows].max(1), M.any_error_score(p[err_rows])),
        "type": M.summary(y, p, 0.5, mask=mask),
        "type_per_class": M.per_class(y, p, NATIVE, 0.5),
    }


def fmt(x, nd=3):
    return "–" if x is None or (isinstance(x, float) and np.isnan(x)) else f"{x:.{nd}f}"


def fmt_ci(v, ci):
    return f"{fmt(v)} ({fmt(ci[0])}–{fmt(ci[1])})"


def write_tables(out, path):
    L = []
    runs, groups = out["runs"], out["groups"]
    L.append("## Table A. Main comparison (official test split, report level, 12 native types)\n")
    L.append("Mean ± SD over seeds when several seeds were run; 95% CI (patient-level bootstrap) for the first seed.\n")
    L.append("| Model | Seeds | Macro-F1 | Micro-F1 | Macro-AUROC | Macro-AUPRC | Any-error AUROC | Any-error AUPRC |")
    L.append("|---|---|---|---|---|---|---|---|")
    for g in sorted(groups.values(), key=lambda g: -g["mean"]["auprc_macro"]):
        r0 = runs[g["runs"][0]]["report"]
        cells = []
        for k in HEADLINE:
            m, s = g["mean"][k], g["sd"][k]
            if len(g["runs"]) > 1:
                cells.append(f"{fmt(m)} ± {fmt(s)}")
            else:
                cells.append(fmt_ci(m, r0["ci95"][k]))
        L.append(f"| {g['name']} | {len(g['runs'])} | " + " | ".join(cells) + " |")

    main = out.get("main_run")
    if main:
        r = runs[main]["report"]
        L.append(f"\n## Table B. Per-type results of the main model ({runs[main]['name']}, threshold 0.5)\n")
        L.append("| Error type | Support | Precision | Recall | F1 | AUROC | AUPRC |")
        L.append("|---|---|---|---|---|---|---|")
        for c in r["native_per_class"]:
            if c["support"] == 0:
                continue
            L.append(f"| {NATIVE_DISPLAY[c['class']]} | {c['support']} | {fmt(c['precision'])} | {fmt(c['recall'])} | "
                     f"{fmt(c['f1'])} | {fmt(c['auroc'])} | {fmt(c['auprc'])} |")
        L.append(f"\n## Table C. Six-group results of the main model\n")
        L.append("| Group | Support | Precision | Recall | F1 | AUROC | AUPRC |")
        L.append("|---|---|---|---|---|---|---|")
        for c in r["group6_per_class"]:
            if c["support"] == 0:
                continue
            L.append(f"| {GROUP6_DISPLAY[c['class']]} | {c['support']} | {fmt(c['precision'])} | {fmt(c['recall'])} | "
                     f"{fmt(c['f1'])} | {fmt(c['auroc'])} | {fmt(c['auprc'])} |")
        L.append("\n## Table C2. Confusion counts of the main model for every error type (threshold 0.5)\n")
        L.append("| Error type | TP | FP | FN | TN | Specificity |")
        L.append("|---|---|---|---|---|---|")
        for c in r["native_per_class"]:
            if c["support"] == 0:
                continue
            L.append(f"| {NATIVE_DISPLAY[c['class']]} | {c['tp']} | {c['fp']} | {c['fn']} | {c['tn']} | "
                     f"{fmt(c['specificity'])} |")
        L.append("\n## Table C3. Per-type thresholds tuned on the validation split (secondary analysis)\n")
        t = r["native_tuned"]
        L.append(f"Macro-F1 {fmt(t['f1_macro'])} (vs. {fmt(r['native']['f1_macro'])} at 0.5); micro-F1 "
                 f"{fmt(t['f1_micro'])} (vs. {fmt(r['native']['f1_micro'])}).\n")
        a = r["any_error"]
        L.append("\n## Table D. Detecting whether a report contains any error (main model)\n")
        L.append("| Operating point | Threshold | Sensitivity | Specificity | Precision (test mix) |")
        L.append("|---|---|---|---|---|")
        for k, lab in (("at_0.5", "Default 0.5"), ("val_sens_0.90", "Sensitivity 0.90 on val"),
                       ("val_sens_0.95", "Sensitivity 0.95 on val")):
            b = a[k]
            L.append(f"| {lab} | {fmt(b['threshold'])} | {fmt(b['sensitivity'])} | {fmt(b['specificity'])} | {fmt(b['precision'])} |")
        L.append("\n## Table E. Expected performance at realistic error prevalence (main model, sensitivity-0.90 operating point)\n")
        L.append("| Prevalence | PPV | NPV | Alerts / 1000 reports | True alerts / 1000 | Missed errors / 1000 |")
        L.append("|---|---|---|---|---|---|")
        for row in a["prevalence_sens_0.90"]:
            L.append(f"| {row['prevalence']:.0%} | {fmt(row['ppv'])} | {fmt(row['npv'])} | {row['alerts_per_1000']:.0f} | "
                     f"{row['true_alerts_per_1000']:.0f} | {row['missed_per_1000']:.1f} |")

    if out["comparisons"]:
        L.append("\n## Table F. Paired bootstrap comparisons against the main model (difference = main − other)\n")
        L.append("| Other model | Δ Macro-F1 (95% CI) | p | Δ Macro-AUPRC (95% CI) | p |")
        L.append("|---|---|---|---|---|")
        for c in out["comparisons"]:
            f, a_ = c["f1_macro"], c["auprc_macro"]
            L.append(f"| {c['other']} | {f['diff']:+.3f} ({f['ci'][0]:+.3f}, {f['ci'][1]:+.3f}) | {f['p_value']:.3f} | "
                     f"{a_['diff']:+.3f} ({a_['ci'][0]:+.3f}, {a_['ci'][1]:+.3f}) | {a_['p_value']:.3f} |")

    for c in out.get("subset_comparisons", []):
        L.append(f"\n## Table H. {c['name']} vs the main model on the same {c['n']} test reports (threshold 0.5)\n")
        L.append("| Model | Macro-F1 | Micro-F1 | Any-error sensitivity | Any-error specificity | Any-error F1 |")
        L.append("|---|---|---|---|---|---|")
        for who, name in (("other", c["name"]), ("main", runs[main]["name"])):
            s_, a_ = c[who]["summary"], c[who]["any"]
            L.append(f"| {name} | {fmt(s_['f1_macro'])} | {fmt(s_['f1_micro'])} | {fmt(a_['sensitivity'])} | "
                     f"{fmt(a_['specificity'])} | {fmt(a_['f1'])} |")

    sent = [k for k, v in runs.items() if v["meta"].get("task") == "sentence"]
    if sent:
        L.append("\n## Table G. Sentence-level error localization\n")
        L.append("| Model | Sentences | Localization AUROC | AUPRC | F1 | Type macro-F1 | Report-level macro-F1 (aggregated) |")
        L.append("|---|---|---|---|---|---|---|")
        for k in sent:
            s, r = runs[k]["sentence"], runs[k]["report"]
            L.append(f"| {runs[k]['name']} | {s['n_sentences']} | {fmt(s['localization']['auroc'])} | "
                     f"{fmt(s['localization']['auprc'])} | {fmt(s['localization']['f1'])} | {fmt(s['type']['f1_macro'])} | "
                     f"{fmt(r['native']['f1_macro'])} |")
    with open(path, "w") as f:
        f.write("\n".join(L) + "\n")


def error_examples(run_dir, data_dir, path, k=5):
    import pandas as pd
    from rexerr.data import build_report_task

    meta = load_meta(run_dir)
    d = load_predictions(run_dir, "test")
    frame, _ = build_report_task(data_dir, "test", meta.get("label_source", "sentence"),
                                 bool(meta.get("include_clean", 1)))
    text = dict(zip(frame["uid"], frame["text"]))
    rows = []
    for j, c in enumerate(NATIVE):
        y, p = d["y"][:, j], d["p"][:, j]
        if y.sum() == 0:
            continue
        fp = np.where(y == 0)[0][np.argsort(-p[y == 0])][:k]
        fn = np.where(y == 1)[0][np.argsort(p[y == 1])][:k]
        for kind, idx in (("false_positive", fp), ("false_negative", fn)):
            for i in idx:
                rows.append({"type": c, "kind": kind, "prob": float(p[i]), "variant": d["variant"][i],
                             "true_types": ";".join(NATIVE[t] for t in np.where(d["y"][i])[0]),
                             "uid": d["uid"][i], "text": text.get(d["uid"][i], "")})
    orig = np.where(d["variant"] == "original")[0]
    s = d["p"][orig].max(1)
    for i in orig[np.argsort(-s)][: 2 * k]:
        rows.append({"type": NATIVE[int(d["p"][i].argmax())], "kind": "false_alarm_on_original_report",
                     "prob": float(d["p"][i].max()), "variant": "original", "true_types": "",
                     "uid": d["uid"][i], "text": text.get(d["uid"][i], "")})
    pd.DataFrame(rows).to_csv(path, index=False)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--runs_dir", default="results/runs")
    ap.add_argument("--out_dir", default="results")
    ap.add_argument("--main", default="biobert", help="substring identifying the main model's runs")
    ap.add_argument("--n_boot", type=int, default=1000)
    ap.add_argument("--data_dir", default=None)
    args = ap.parse_args()

    runs = {}
    for rd in sorted(glob.glob(os.path.join(args.runs_dir, "*"))):
        if not os.path.exists(os.path.join(rd, "preds_test.npz")):
            continue
        key = os.path.basename(rd)
        meta = load_meta(rd)
        val, test = load_predictions(rd, "val"), load_predictions(rd, "test")
        entry = {"dir": rd, "meta": {k: v for k, v in meta.items() if k != "history"}, "name": display(meta)}
        if meta.get("task") == "sentence":
            entry["sentence"] = evaluate_sentence_level(test)
            val, test = aggregate_sentences(val), aggregate_sentences(test)
        entry["report"] = evaluate_report_level(val, test, args.n_boot)
        runs[key] = entry
        print(f"[eval] {key}: macro-F1 {entry['report']['native']['f1_macro']:.4f} "
              f"macro-AUPRC {entry['report']['native']['auprc_macro']:.4f}", flush=True)

    # Seed aggregation: same model/task/labels -> one group.
    groups = defaultdict(list)
    for key, e in runs.items():
        m = e["meta"]
        gid = "|".join(str(m.get(x)) for x in ("model", "task", "label_source", "include_clean", "loss"))
        gid += "|" + str((m.get("args") or {}).get("tag", ""))
        groups[gid].append(key)
    out_groups = {}
    for gid, keys in groups.items():
        vals = {k: [] for k in HEADLINE}
        for key in keys:
            r = runs[key]["report"]
            for k in HEADLINE[:4]:
                vals[k].append(r["native"][k])
            vals["any_auroc"].append(r["any_error"]["at_0.5"]["auroc"])
            vals["any_auprc"].append(r["any_error"]["at_0.5"]["auprc"])
        out_groups[gid] = {"name": runs[keys[0]]["name"], "runs": keys,
                           "mean": {k: float(np.mean(v)) for k, v in vals.items()},
                           "sd": {k: float(np.std(v, ddof=1)) if len(v) > 1 else 0.0 for k, v in vals.items()}}

    # main model: standard configuration (report task, BCE loss, controls, sentence labels), lowest seed
    def is_standard(m):
        return (m.get("task") == "report" and m.get("loss", "bce") in (None, "bce")
                and str(m.get("include_clean", 1)) == "1" and m.get("label_source", "sentence") == "sentence")
    main_keys = [k for k in runs if args.main in k and is_standard(runs[k]["meta"])]
    main_run = min(main_keys, key=lambda k: (runs[k]["meta"].get("seed", 0), k)) if main_keys else None
    comparisons = []
    if main_run:
        d_main = load_predictions(runs[main_run]["dir"], "test")
        f1 = lambda y, p: M.summary(y, p, 0.5, mask=M.active_classes(d_main["y"]))["f1_macro"]  # noqa: E731
        ap_ = lambda y, p: M.summary(y, p, 0.5, mask=M.active_classes(d_main["y"]))["auprc_macro"]  # noqa: E731
        for gid, g in out_groups.items():
            other = sorted(g["runs"])[0]
            if other == main_run or main_run in g["runs"] or runs[other]["meta"].get("task") != "report":
                continue
            d_o = load_predictions(runs[other]["dir"], "test")
            if not np.array_equal(d_o["uid"], d_main["uid"]):
                continue
            comparisons.append({
                "other": g["name"],
                "f1_macro": M.paired_bootstrap(f1, d_main["y"], d_main["p"], d_o["p"], d_main["subject_id"], args.n_boot),
                "auprc_macro": M.paired_bootstrap(ap_, d_main["y"], d_main["p"], d_o["p"], d_main["subject_id"], args.n_boot),
            })

    subset = []
    if main_run:
        pos = {u: i for i, u in enumerate(d_main["uid"])}
        for key, e in runs.items():
            d_o = load_predictions(e["dir"], "test")
            if key == main_run or e["meta"].get("task") != "report" or len(d_o["uid"]) >= len(d_main["uid"]):
                continue
            if not all(u in pos for u in d_o["uid"]):
                continue
            idx = np.array([pos[u] for u in d_o["uid"]])
            y = d_main["y"][idx].astype(int)
            mask = M.active_classes(y)
            row = {"name": e["name"], "n": int(len(idx))}
            for who, p in (("other", d_o["p"]), ("main", d_main["p"][idx])):
                row[who] = {"summary": M.summary(y, p, 0.5, mask=mask), "any": M.binary(y.max(1), M.any_error_score(p)),
                            "per_class": M.per_class(y, p, NATIVE, 0.5)}
            subset.append(row)

    out = {"runs": runs, "groups": out_groups, "main_run": main_run, "comparisons": comparisons,
           "subset_comparisons": subset}
    os.makedirs(args.out_dir, exist_ok=True)
    with open(os.path.join(args.out_dir, "summary.json"), "w") as f:
        json.dump(out, f, indent=1, default=lambda o: o.tolist() if hasattr(o, "tolist") else str(o))
    write_tables(out, os.path.join(args.out_dir, "tables.md"))
    if main_run and args.data_dir:
        error_examples(runs[main_run]["dir"], args.data_dir, os.path.join(args.out_dir, "error_examples.csv"))
    print(open(os.path.join(args.out_dir, "tables.md")).read())


if __name__ == "__main__":
    main()


### Data (ReXErr-v1, open access, ODC-By 1.0)


In [ ]:
import glob, os, shutil, subprocess, time, zipfile
import requests

def kaggle_token():
    token = os.environ.get("KAGGLE_API_TOKEN")
    if not token and IN_COLAB:
        try:
            from google.colab import userdata
            token = userdata.get("KAGGLE_API_TOKEN")
        except Exception:
            token = None
    return token

def fetch_from_kaggle(dataset, token, dest):
    # requests drops the Authorization header on the redirect to the storage server
    with requests.get(f"https://www.kaggle.com/api/v1/datasets/download/{dataset}",
                      headers={"Authorization": f"Bearer {token}"}, stream=True, timeout=120) as r:
        if r.status_code != 200:
            raise RuntimeError(f"Kaggle download failed with HTTP {r.status_code}")
        with open(dest, "wb") as f:
            shutil.copyfileobj(r.raw, f, 1 << 20)

def fetch_from_physionet(dest):
    url = "https://physionet.org/content/rexerr-v1/get-zip/1.0.0/"
    for attempt in range(40):  # PhysioNet often resets or stalls long downloads; curl -C - resumes
        print("downloading ReXErr-v1 from PhysioNet, attempt", attempt + 1, flush=True)
        if subprocess.run(f'curl -sS -L -C - --speed-limit 2000 --speed-time 120 -o "{dest}" "{url}"',
                          shell=True).returncode == 0:
            return
        time.sleep(5)
    raise RuntimeError("PhysioNet download failed")

def download_rexerr():
    if os.path.exists(f"{DATA}/ReXErr-report-level/ReXErr-report-level_test.csv"):
        print("data already present"); return
    # Kaggle: a copy of ReXErr-v1 added as an input (Add Input > Datasets) is used instead of downloading
    mirror = glob.glob("/kaggle/input/**/ReXErr-report-level/ReXErr-report-level_test.csv", recursive=True)
    if mirror:
        root = os.path.dirname(os.path.dirname(mirror[0]))
        for d in ("ReXErr-report-level", "ReXErr-sentence-level"):
            shutil.copytree(f"{root}/{d}", f"{DATA}/{d}", dirs_exist_ok=True)
        print("data copied from", root); return
    zips = [z for z in glob.glob("/kaggle/input/**/*.zip", recursive=True)  # not rexerr_results.zip of earlier runs
            if any("ReXErr-report-level/" in n for n in zipfile.ZipFile(z).namelist())]
    saved = f"{CACHE}/rexerr-v1-1.0.0.zip"  # verified archive kept between sessions (on Drive in Colab)
    if zips:
        archive = zips[0]
    elif os.path.exists(saved + ".done") and os.path.exists(saved):  # the marker alone is not enough
        archive = saved
    else:
        archive = "/tmp/rexerr-download.zip"  # local disk: resuming a partial file on Drive leaves duplicates
        token = kaggle_token()
        if KAGGLE_DATASET and token:
            print("downloading ReXErr-v1 from Kaggle dataset", KAGGLE_DATASET, flush=True)
            fetch_from_kaggle(KAGGLE_DATASET, token, archive)
        else:
            fetch_from_physionet(archive)
        subprocess.run(f'unzip -tq "{archive}"', shell=True, check=True)
        if os.path.dirname(saved) != "/tmp":
            shutil.copy(archive, saved)
            open(saved + ".done", "w").close()
    shutil.rmtree("/tmp/rexerr_unzip", ignore_errors=True)
    subprocess.run(f'unzip -q -o "{archive}" -d /tmp/rexerr_unzip', shell=True, check=True)
    for root, dirs, files in os.walk("/tmp/rexerr_unzip"):  # the zip has one top-level folder
        if "ReXErr-report-level" in dirs:
            subprocess.run(f'cp -r "{root}"/* "{DATA}"/', shell=True, check=True); break
    print(sorted(os.listdir(DATA)))
download_rexerr()


### Experiments


In [ ]:
import json, os, shlex, subprocess, time
FAILED = []

def run(cmd, name=None):
    # A failed run is recorded and the notebook carries on, so the results of the other runs are still packaged.
    print("$", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, cwd=CODE, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        if "UNEXPECTED" in line or "can be ignored" in line:
            continue
        print(line, end="", flush=True)
    if p.wait() != 0:
        FAILED.append(name or cmd)
        print(f"[failed] {name or cmd}", flush=True)
        return False
    return True

def train(name, model, extra=""):
    out = f"{RUNS}/{name}"
    if os.path.exists(f"{out}/preds_test.npz"):
        print(f"[skip] {name} already finished"); return
    t = time.time()
    run(f"python scripts/train_encoder.py --data_dir {shlex.quote(DATA)} --out_dir {shlex.quote(out)} "
        f"--model {model} --epochs {EPOCHS} --batch_size {max(1, BATCH_SIZE // N_GPU)} {extra}", name)
    print(f"[time] {name}: {(time.time() - t) / 60:.1f} min")

run(f"python scripts/audit.py --data_dir {shlex.quote(DATA)} --out {shlex.quote(WORK)}/audit.json "
    f"--tokenizer dmis-lab/biobert-v1.1 > /dev/null", "audit")


In [ ]:
if "main" in RUN_GROUPS:
    for seed in (0, 1, 2):
        train(f"biobert_s{seed}", "dmis-lab/biobert-v1.1", f"--seed {seed}")


In [ ]:
if "encoders" in RUN_GROUPS:
    train("bert-base-cased_s0", "google-bert/bert-base-cased")
    train("clinicalbert_s0", "emilyalsentzer/Bio_ClinicalBERT")
    train("biomedbert_s0", "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext")
    train("cxrbert_s0", "microsoft/BiomedVLP-CXR-BERT-general")
    train("bluebert_s0", "bionlp/bluebert_pubmed_mimic_uncased_L-12_H-768_A-12")
    train("radbert_s0", "UCSD-VA-health/RadBERT-RoBERTa-4m")


In [ ]:
if "extras" in RUN_GROUPS:
    # error localization: (sentence, report) pairs from a patient-level subset of the training split
    train("biobert_sentence_s0", "dmis-lab/biobert-v1.1", "--task sentence --max_train_studies 40000 --epochs 1")
    # ablation: no error-free reports during training
    train("biobert_noclean_s0", "dmis-lab/biobert-v1.1", "--include_clean 0")
    # class-imbalance strategies
    train("biobert_weighted_s0", "dmis-lab/biobert-v1.1", "--loss weighted")
    train("biobert_focal_s0", "dmis-lab/biobert-v1.1", "--loss focal")
    # zero-shot LLMs; whole test split when the GPUs hold the model in 16-bit (>= 20 GB in total, e.g. T4 x2)
    big = GPU_MEM_TOTAL >= 20e9
    for key, model in (("llm_qwen2.5-7b", "Qwen/Qwen2.5-7B-Instruct"),
                       ("llm_llama3.1-8b", "unsloth/Meta-Llama-3.1-8B-Instruct"),  # ungated copy of Meta's weights
                       ("llm_qwen3-8b", "Qwen/Qwen3-8B")):
        out = f"{RUNS}/{key}"
        if not os.path.exists(f"{out}/preds_test.npz"):
            run(f"python scripts/llm_zeroshot.py --data_dir {shlex.quote(DATA)} --out_dir {shlex.quote(out)} "
                f"--model {model} --n_studies {0 if big else 500} --batch_size {16 if big else 8}", key)


In [ ]:
if "tfidf" in RUN_GROUPS:
    ram = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES")
    if os.path.exists(f"{RUNS}/tfidf_sentence_clean1/preds_test.npz"):
        print("[skip] tfidf already finished")
    elif ram > 24e9:
        run(f"python scripts/baselines.py --data_dir {shlex.quote(DATA)} --runs_dir {shlex.quote(RUNS)} --model tfidf",
            "tfidf")
    else:
        print(f"only {ram / 1e9:.0f} GB RAM: full-data TF-IDF needs > 24 GB (Kaggle CPU sessions have 30 GB)")


### Package the results


In [ ]:
import glob, os, shutil, zipfile
zpath = f"{WORK}/rexerr_results.zip"
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    if os.path.exists(f"{WORK}/audit.json"):
        z.write(f"{WORK}/audit.json", "audit.json")
    for f in glob.glob(f"{RUNS}/*/*"):
        if os.path.basename(f) in ("preds_val.npz", "preds_test.npz", "meta.json", "raw_outputs.jsonl"):
            z.write(f, os.path.relpath(f, WORK))
print("finished runs:", sorted(os.path.basename(os.path.dirname(f)) for f in glob.glob(f"{RUNS}/*/preds_test.npz")))
print("failed runs:", FAILED or "none")
print("wrote", zpath, f"({os.path.getsize(zpath) / 1e6:.1f} MB)")
if IN_KAGGLE:
    shutil.copy(zpath, "/kaggle/working/rexerr_results.zip")
    print("Download rexerr_results.zip from the Output tab of this version.")
elif IN_COLAB:
    print("Get it from Google Drive: MyDrive/rexerr_experiments/rexerr_results.zip")
    if UNASSIGN_WHEN_DONE:
        from google.colab import runtime
        runtime.unassign()  # stop consuming compute units
